# My Version

In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models
from tqdm.auto import tqdm
from sklearn.metrics import f1_score, classification_report
from sklearn.model_selection import train_test_split
import numpy as np
import json

# 1. Setup Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 2. Data Transformation
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# 3. Load Dataset & Stratified Splitting
# Pastikan path ini sesuai dengan struktur di Kaggle Anda
data_dir = '/kaggle/input/datasets/abdallahalidev/plantvillage-dataset/color' 
full_dataset = datasets.ImageFolder(root=data_dir, transform=transform)

# Mengambil mapping nama folder asli
class_names = full_dataset.classes
num_classes = len(class_names)

# Simpan mapping ke JSON agar bisa digunakan saat deployment nanti
idx_to_class = {i: name for i, name in enumerate(class_names)}
with open('class_mapping.json', 'w') as f:
    json.dump(idx_to_class, f)

# Dapatkan label untuk stratifikasi
labels = full_dataset.targets

# Stratified Split (80% Train, 20% Val)
train_indices, val_indices = train_test_split(
    range(len(labels)),
    test_size=0.2,
    stratify=labels,
    random_state=42
)

train_dataset = Subset(full_dataset, train_indices)
val_dataset = Subset(full_dataset, val_indices)

# 4. Data Loader
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

print(f"Total Kelas: {num_classes}")
print(f"Training Images: {len(train_dataset)}")
print(f"Validation Images: {len(val_dataset)}")

# 5. Build Model (EfficientNet-B0)
model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
num_ftrs = model.classifier[1].in_features
model.classifier[1] = nn.Linear(num_ftrs, num_classes)
model = model.to(device)

# 6. Loss & Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

# 7. Training & Validation Loop
num_epochs = 5

for epoch in range(num_epochs):
    # --- PHASE TRAINING ---
    model.train()
    train_loss = 0.0
    t_preds, t_labels = [], []
    
    pbar_train = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Train]", unit="batch")
    for images, labels in pbar_train:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item() * images.size(0)
        _, pred = torch.max(outputs, 1)
        
        t_preds.extend(pred.cpu().numpy())
        t_labels.extend(labels.cpu().numpy())
        
        curr_f1 = f1_score(t_labels, t_preds, average='macro')
        pbar_train.set_postfix({"loss": f"{loss.item():.3f}", "f1": f"{curr_f1:.3f}"})

    # --- PHASE VALIDATION ---
    model.eval()
    val_loss = 0.0
    v_preds, v_labels = [], []
    
    print(f"Menjalankan Validasi Epoch {epoch+1}...")
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * images.size(0)
            _, pred = torch.max(outputs, 1)
            
            v_preds.extend(pred.cpu().numpy())
            v_labels.extend(labels.cpu().numpy())

    final_train_f1 = f1_score(t_labels, t_preds, average='macro')
    final_val_f1 = f1_score(v_labels, v_preds, average='macro')
    
    print(f"\n>> Epoch {epoch+1} Selesai")
    print(f"Train Loss: {train_loss/len(train_dataset):.4f} | Train F1: {final_train_f1:.4f}")
    print(f"Val Loss: {val_loss/len(val_dataset):.4f} | Val F1: {final_val_f1:.4f}\n")

# 8. Menampilkan Contoh Hasil Prediksi dengan Nama Folder
print("--- Contoh Prediksi pada Data Validasi ---")
model.eval()
with torch.no_grad():
    sample_images, sample_labels = next(iter(val_loader))
    sample_images = sample_images.to(device)
    outputs = model(sample_images)
    _, preds = torch.max(outputs, 1)
    
    for i in range(5): # Tampilkan 5 contoh
        pred_name = class_names[preds[i].item()]
        true_name = class_names[sample_labels[i].item()]
        print(f"Data {i+1}: Prediksi -> {pred_name} | Asli -> {true_name}")

# 9. Save Model
torch.save({
    'model_state_dict': model.state_dict(),
    'class_names': class_names
}, 'efficientnet_plantvillage_final.pth')

print("\nModel dan Nama Kelas Berhasil Disimpan!")

Using device: cuda
Total Kelas: 38
Training Images: 43444
Validation Images: 10861


Epoch 1/5 [Train]:   0%|          | 0/1358 [00:00<?, ?batch/s]

Menjalankan Validasi Epoch 1...

>> Epoch 1 Selesai
Train Loss: 0.3809 | Train F1: 0.8870
Val Loss: 0.0269 | Val F1: 0.9878



Epoch 2/5 [Train]:   0%|          | 0/1358 [00:00<?, ?batch/s]

Menjalankan Validasi Epoch 2...

>> Epoch 2 Selesai
Train Loss: 0.0391 | Train F1: 0.9845
Val Loss: 0.0214 | Val F1: 0.9877



...

Menjalankan Validasi Epoch 3...

>> Epoch 3 Selesai
Train Loss: 0.0224 | Train F1: 0.9906
Val Loss: 0.0201 | Val F1: 0.9898



Epoch 4/5 [Train]:   0%|          | 0/1358 [00:00<?, ?batch/s]

Menjalankan Validasi Epoch 4...

>> Epoch 4 Selesai
Train Loss: 0.0176 | Train F1: 0.9926
Val Loss: 0.0141 | Val F1: 0.9942



Epoch 5/5 [Train]:   0%|          | 0/1358 [00:00<?, ?batch/s]

Menjalankan Validasi Epoch 5...

>> Epoch 5 Selesai
Train Loss: 0.0134 | Train F1: 0.9942
Val Loss: 0.0115 | Val F1: 0.9957

--- Contoh Prediksi pada Data Validasi ---
Data 1: Prediksi -> Orange___Haunglongbing_(Citrus_greening) | Asli -> Orange___Haunglongbing_(Citrus_greening)
Data 2: Prediksi -> Soybean___healthy | Asli -> Soybean___healthy
Data 3: Prediksi -> Grape___Black_rot | Asli -> Grape___Black_rot
Data 4: Prediksi -> Soybean___healthy | Asli -> Soybean___healthy
Data 5: Prediksi -> Cherry_(including_sour)___Powdery_mildew | Asli -> Cherry_(including_sour)___Powdery_mildew

Model dan Nama Kelas Berhasil Disimpan!


In [6]:
import os
file_path = 'efficientnet_b0_stratified.pth'
if os.path.exists(file_path):
    print(f"File ditemukan! Ukuran: {os.path.getsize(file_path) / (1024*1024):.2f} MB")
else:
    print("File TIDAK ditemukan. Cek kembali path penyimpanan Anda.")

File ditemukan! Ukuran: 15.76 MB


In [7]:
!zip -r models_backup.zip /kaggle/working/

  adding: kaggle/working/ (stored 0%)
  adding: kaggle/working/.virtual_documents/ (stored 0%)
  adding: kaggle/working/.virtual_documents/__notebook_source__.ipynb (deflated 71%)
  adding: kaggle/working/efficientnet_b0_stratified.pth (deflated 8%)


In [8]:
from IPython.display import FileLink
# Ganti nama file sesuai dengan file zip Anda
FileLink(r'models_backup.zip')

/kaggle/working/models_backup.zip